<a href="https://colab.research.google.com/github/Thomas-18/lab/blob/main/MLE%26MAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Set random seed for reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")



In [5]:
sns.set_context("notebook")
#Load dataset
data=load_breast_cancer()
x=pd.DataFrame(data.data,columns=data.feature_names)
y=data.target
print(f"Dataset Shape: {x.shape}")
print(f"Class distribution: {np.bincount(y)} (0: malignant, 1: Benign)")

# Train-Test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#standardize features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
print("Preprocessing complete.")


Dataset Shape: (569, 30)
Class distribution: [212 357] (0: malignant, 1: Benign)


In [13]:
mle_model = LogisticRegression(penalty=None, max_iter=10000)
mle_model.fit(x_train, y_train)

map_12_model = LogisticRegression(penalty='l2', C=10, max_iter=10000)
map_12_model.fit(x_train, y_train)

map_l1_model = LogisticRegression(penalty='l1', solver='saga' , C=1.0, max_iter=10000)
map_l1_model.fit(x_train, y_train)

weights_df =pd.DataFrame({
    'Feature' :['Intercept'] + list(data.feature_names),
    'MLE' : np.insert(mle_model.coef_[0], 0, mle_model.intercept_[0]),
    'MAP_L2 (Gaussian)' : np.insert(map_12_model.coef_[0], 0, map_12_model.intercept_[0]),
    'MAP_L1 (Lasso)' : np.insert(map_l1_model.coef_[0], 0, map_l1_model.intercept_[0])
})

def evaluation(model, x, y, name):
  pred_labels = model.predict(x)
  pred_proba = model.predict_proba(x)
  return {
      'Model' : name,
      'Accuracy' : accuracy_score(y, pred_labels),
      'Precision' : precision_score(y, pred_labels),
      'Recall' : recall_score(y, pred_labels),
      'F1 Score' : f1_score(y, pred_labels),
      'ROC AUC' : roc_auc_score(y, pred_proba[:, 1])
  }

results =[
    evaluation(mle_model, x_test, y_test, 'MLE (No Regularization)'),
    evaluation(map_12_model, x_test, y_test, 'MAP (L2 Regularization)'),
    evaluation(map_l1_model, x_test, y_test, 'MAP (L1 Regularization)')
]

metrics_df = pd.DataFrame(results)
print(weights_df)
print("\n")
print(metrics_df)

                    Feature         MLE  MAP_L2 (Gaussian)  MAP_L1 (Lasso)
0                 Intercept  -63.532600          -0.320674        0.299011
1               mean radius    9.433096          -0.033622        0.000000
2              mean texture  -16.986388          -0.091143        0.000000
3            mean perimeter   40.020926           0.180414        0.000000
4                 mean area   10.890150          -0.143463        0.000000
5           mean smoothness    5.033443           0.043645        0.000000
6          mean compactness  274.940098           2.701495        0.000000
7            mean concavity -134.838158          -1.585240        0.000000
8       mean concave points -266.184736          -3.280577       -2.195840
9             mean symmetry   40.996505           0.848441        0.053361
10   mean fractal dimension -168.017946          -1.005232        0.000000
11             radius error -259.639462          -3.500717       -2.446271
12            texture err